In [ ]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import Counter
import re

nltk.download('vader_lexicon')
nltk.download('stopwords')
from nltk.corpus import stopwords

from google.colab import files
uploaded = files.upload()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Saving customer_reviews_cleaned.csv to customer_reviews_cleaned.csv


In [ ]:
df = pd.read_csv('customer_reviews_cleaned.csv', header=None, names=['ReviewID','CustomerID','ProductID','ReviewDate','Rating','ReviewText'])
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,1,5,2025-03-19,2,"Good quality, but could be cheaper."
1,2,1,2,2025-11-05,5,Five stars for the quick delivery.
2,3,1,8,2023-07-05,5,The quality is top-notch.
3,4,1,19,2023-05-07,4,Customer support was very helpful.
4,5,1,2,2023-09-30,5,Shipping was fast and the item was well-packa...


In [ ]:
sia = SentimentIntensityAnalyzer()
df['compound'] = df['ReviewText'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

def categorize_sentiment(row):
    compound = row['compound']
    rating = row['Rating']
    if compound >= 0.05:
        return 'Positive' if rating >= 4 else 'Mixed Positive'
    elif compound <= -0.05:
        return 'Negative' if rating <= 2 else 'Mixed Negative'
    else:
        return 'Neutral'

df['SentimentCategory'] = df.apply(categorize_sentiment, axis=1)
df['SentimentCategory'].value_counts()

,count
SentimentCategory,
Positive,565
Neutral,469
Negative,354
Mixed Positive,58


In [ ]:
df.to_csv('customer_reviews_sentiment.csv', index=False)
files.download('customer_reviews_sentiment.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
negative_df = df[df['SentimentCategory'].isin(['Negative', 'Mixed Negative'])].copy()

stop_words = set(stopwords.words('english'))

def get_words(text):
    words = re.findall(r'\b[a-z]+\b', str(text).lower())
    return [w for w in words if w not in stop_words and len(w) > 2]

all_words = []
for text in negative_df['ReviewText']:
    all_words.extend(get_words(text))

word_freq = Counter(all_words)
word_freq.most_common(20)

[('product', 176),
 ('experience', 94),
 ('disappointed', 52),
 ('performance', 52),
 ('average', 50),
 ('nothing', 50),
 ('special', 50),
 ('okay', 49),
 ('instructions', 49),
 ('unclear', 49),
 ('bad', 44),
 ('stopped', 42),
 ('working', 42),
 ('month', 42),
 ('satisfied', 41),
 ('broke', 41),
 ('week', 41),
 ('terrible', 40),
 ('customer', 40),
 ('service', 40)]

In [ ]:
theme_keywords = {
    'Unmet Expectations': ['expectations', 'meet', 'average', 'nothing', 'special'],
    'Product Performance': ['performance', 'disappointed', 'stopped', 'working', 'bad'],
    'Delivery': ['arrived', 'late', 'delivery'],
    'Product Cost': ['worth', 'money', 'cheaper'],
    'Durability': ['broke', 'week', 'month'],
    'Usability': ['instructions', 'unclear', 'okay'],
    'Customer Service': ['terrible', 'customer', 'service', 'buy', 'again'],
    'General Dissatisfaction': ['color', 'different', 'shown']
}

def assign_theme(text):
    text_lower = str(text).lower()
    matched = [theme for theme, kws in theme_keywords.items() if any(k in text_lower for k in kws)]
    return ', '.join(matched) if matched else 'Other'

negative_df['Theme'] = negative_df['ReviewText'].apply(assign_theme)
negative_df['Theme'].value_counts()

,count
Theme,
Product Performance,96
Unmet Expectations,50
Usability,49
"Product Performance, Durability",42
Durability,41
Customer Service,40
Product Cost,36


In [ ]:
negative_df[['ReviewID','CustomerID','ProductID','ReviewDate','Rating','ReviewText','SentimentCategory','Theme']].to_csv('negative_reviews_issues.csv', index=False)
files.download('negative_reviews_issues.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>